In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import pytz
sys.path.append(os.path.abspath(".."))
import src.raw_preprocessing as rp
import src.feature_engineering as fe
import src.create_dataset as cd
from datetime import date, datetime, timedelta
from vacances_scolaires_france import SchoolHolidayDates

import openmeteo_requests
import requests_cache
from retry_requests import retry
from joblib import load

In [134]:
conso = pd.read_parquet("../data/final_datasets/datasets_linear_models/conso_v3_linear.parquet")

In [135]:
conso.columns

Index(['Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '59T', '75T', '13T', '33T', 'T', 'U', 'FF', 'PMER', 'RR1',
       'year', 'month', 'hour', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin',
       'month_cos', 'lagged_1', 'lagged_2', 'lagged_48', 'lagged_336',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
  

In [24]:
# This function below is not the same as the one that we use for our pipeline. 
# Unlike this function, the one we use in our pipeline deletes the dataset in its directory before 
# downloading a new dataset

cd.download_monthly_data()

the download of conso_energie_2026.zip has started


In [3]:
df = rp.conso_preprocess(Path("../data/conso/real_time_conso/"))

df = df[df["Heures"].apply(lambda x: x.minute in {00, 30})]

df = df.reset_index(drop=True)
df.loc[len(df)] = None

In [4]:
df.columns

Index(['Date', 'Heures', 'Consommation'], dtype='object')

In [5]:
df

,Date,Heures,Consommation
0,2026-07-01,00:00:00,46807.0
1,2026-07-01,00:30:00,45014.0
2,2026-07-01,01:00:00,43059.0
3,2026-07-01,01:30:00,42755.0
4,2026-07-01,02:00:00,41666.0
...,...,...,...
3100,2026-09-03,14:00:00,49031.0
3101,2026-09-03,14:30:00,49703.0
3102,2026-09-03,15:00:00,49080.0
3103,2026-09-03,15:30:00,48552.0


In [6]:
df = fe.lagged_consumption(df)

In [7]:
last_date = df["Date"].iloc[-2]
last_time = df["Heures"].iloc[-2]

last_datetime = datetime.combine(
    pd.to_datetime(last_date).date(),
    last_time
)

new_datetime = last_datetime + timedelta(minutes=30)

df.loc[len(df)-1, "Date"] = new_datetime.strftime("%Y-%m-%d")
df.loc[len(df)-1, "Heures"] = new_datetime.time()

In [8]:
t = df["Heures"].iloc[-2]

new_time = (
    datetime.combine(datetime.today(), t)
    + timedelta(minutes=30)
).time()

df.loc[len(df)-1, "Heures"] = new_time

df.loc[len(df)-1, "Date"] = datetime.today().strftime('%Y-%m-%d')

In [9]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-07-01,00:00:00,46807.0,NaN,NaN,NaN,NaN
1,2026-07-01,00:30:00,45014.0,46807.0,NaN,NaN,NaN
2,2026-07-01,01:00:00,43059.0,45014.0,46807.0,NaN,NaN
3,2026-07-01,01:30:00,42755.0,43059.0,45014.0,NaN,NaN
4,2026-07-01,02:00:00,41666.0,42755.0,43059.0,NaN,NaN
...,...,...,...,...,...,...,...
3100,2026-09-03,14:00:00,49031.0,48312.0,48952.0,49020.0,51467.0
3101,2026-09-03,14:30:00,49703.0,49031.0,48312.0,49789.0,52070.0
3102,2026-09-03,15:00:00,49080.0,49703.0,49031.0,48947.0,51525.0
3103,2026-09-03,15:30:00,48552.0,49080.0,49703.0,48081.0,51171.0


#### Adding the name of the holidays

In [13]:
from pprint import pprint
today = date(2026, 5, 16).isoformat()

url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

params = {
    "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone C'"
}

response = requests.get(url, params=params)

In [11]:
#Adding the infos about the holidays
today = date.today().isoformat()
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

zones = ['A', 'B', 'C']
vacances = ['vacances de la toussaint', 'vacances de noël', "vacances d'hiver",'vacances de printemps', "vacances d'été"]
feries = [
    "jour de l'an",
    "lundi de pâques",
    "fête du travail",
    "victoire 1945",
    "ascension",
    "lundi de pentecôte",
    "fête nationale",
    "assomption",
    "toussaint",
    "armistice",
    "noël",
    "pont de l'ascension",
]

#finished_with_holidays = False
df.loc[:, "Zone_A"] = 0
df.loc[:, "Zone_B"] = 0
df.loc[:, "Zone_C"] = 0

df.loc[:, "public_holidays"] = 0
df.loc[:, "Vacances de la Toussaint"] = 0
df.loc[:, "Vacances de Noël"] = 0
df.loc[:, "Vacances d'Hiver"] = 0
df.loc[:, "Vacances de Printemps"] = 0
df.loc[:, "Vacances d'Été"] = 0

for z in zones:
    params = {
        "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone {z}'"
    }
    response = requests.get(url, params=params)
    data = response.json()
    print(data)
    # If no holidays we set the columns with the value 0
    if data["total_count"] == 0:
        continue

    else : 
        
        for event in data["results"]:
            # We check if it's school holidays
            if event["description"].lower() in vacances:
                df.loc[:, f"Zone_{z}"] = 1
                if event["description"].lower() == "vacances de la toussaint":
                    df.loc[:, "Vacances de la Toussaint"] = 1
                    
                elif event["description"].lower() == "vacances de noël":
                    df.loc[:, "Vacances de Noël"] = 1
                    
                elif event["description"].lower() == "vacances d'hiver":
                    df.loc[:, "Vacances d'Hiver"] = 1
                    
                elif event["description"].lower() == "vacances de printemps":
                    df.loc[:, "Vacances de Printemps"] = 1

                elif event["description"].lower() == "vacances d'été":
                    df.loc[:, "Vacances d'Été"] = 1
            # Or if it's public holidays (jours fériés)
            elif event["description"].lower() in feries:
                df.loc[:, "public_holidays"] = 1

{'total_count': 0, 'results': []}
{'total_count': 0, 'results': []}
{'total_count': 0, 'results': []}


In [70]:
df["Consommation"] = pd.to_numeric(df["Consommation"], errors="coerce")
df["Consommation"] = df["Consommation"].interpolate()

In [71]:
df = fe.date_and_hour_pred(df)
df = fe.cyclical_encoding(df)
df = fe.rolling_window(df)
df = fe.lagged_trend(df)
df = fe.seasons_linear(df)

In [72]:
df

,Consommation,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,...,rolling_std_7d,rolling_max_24h,rolling_min_24h,consumption_diff_1,consumption_diff_48,consumption_pct_change_1,consumption_pct_change_48,season_Spring,season_Summer,season_Winter
0,46807.0,NaN,NaN,NaN,NaN,1,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
1,45014.0,46807.0,NaN,NaN,NaN,1,1,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
2,43059.0,45014.0,46807.0,NaN,NaN,1,1,1,0,0,...,NaN,NaN,NaN,-1793.0,NaN,-0.038306,NaN,0,1,0
3,42755.0,43059.0,45014.0,NaN,NaN,1,1,1,0,0,...,NaN,NaN,NaN,-1955.0,NaN,-0.043431,NaN,0,1,0
4,41666.0,42755.0,43059.0,NaN,NaN,1,1,1,0,0,...,NaN,NaN,NaN,-304.0,NaN,-0.007060,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2590,40331.0,39917.0,39907.0,39511.0,43149.0,1,1,1,0,0,...,4959.276057,41376.0,30287.0,10.0,493.0,0.000251,0.012505,0,1,0
2591,39406.0,40331.0,39917.0,38919.0,41915.0,1,1,1,0,0,...,4958.058938,41376.0,30287.0,414.0,820.0,0.010372,0.020754,0,1,0
2592,38804.0,39406.0,40331.0,38131.0,40865.0,1,1,1,0,0,...,4958.593814,41376.0,30287.0,-925.0,487.0,-0.022935,0.012513,0,1,0
2593,37346.0,38804.0,39406.0,36782.0,39318.0,1,1,1,0,0,...,4960.049346,41376.0,30287.0,-602.0,673.0,-0.015277,0.017650,0,1,0


In [73]:
df["full_date"].dt.date

0       2026-07-01
1       2026-07-01
2       2026-07-01
3       2026-07-01
4       2026-07-01
           ...    
2590    2026-08-23
2591    2026-08-23
2592    2026-08-24
2593    2026-08-24
2594    2026-08-24
Name: full_date, Length: 2595, dtype: object

In [80]:
france_tz = pytz.timezone("Europe/Paris")
current_time = datetime.now(france_tz).replace(second=0, microsecond=0)

In [81]:
current_time

datetime.datetime(2026, 8, 24, 0, 47, tzinfo=<DstTzInfo 'Europe/Paris' CEST+2:00:00 DST>)

In [84]:
df["full_date"]

0      2026-07-01 00:00:00
1      2026-07-01 00:30:00
2      2026-07-01 01:00:00
3      2026-07-01 01:30:00
4      2026-07-01 02:00:00
               ...        
2590   2026-08-23 23:00:00
2591   2026-08-23 23:30:00
2592   2026-08-24 00:00:00
2593   2026-08-24 00:30:00
2594   2026-08-24 01:00:00
Name: full_date, Length: 2595, dtype: datetime64[ns]

In [85]:
hist_today = df[df["full_date"].dt.date == current_time.date()][["full_date", "Consommation"]].iloc[:-1, :]

In [86]:
hist_today

,full_date,Consommation
2592,2026-08-24 00:00:00,38804.0
2593,2026-08-24 00:30:00,37346.0


In [43]:
df = df.drop(["Consommation"], axis=1)

In [44]:
df.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter'],
      dtype='object')

In [45]:
row = df.iloc[-1]

pred = pd.DataFrame([row] * 10)

pred["full_date"] = row["full_date"] + pd.to_timedelta(range(10), unit="m") * 30
pred = pred.reset_index(drop=True)

pred.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter'],
      dtype='object')

In [48]:
pred["full_date"]

0   2026-08-23 00:00:00
1   2026-08-23 00:30:00
2   2026-08-23 01:00:00
3   2026-08-23 01:30:00
4   2026-08-23 02:00:00
5   2026-08-23 02:30:00
6   2026-08-23 03:00:00
7   2026-08-23 03:30:00
8   2026-08-23 04:00:00
9   2026-08-23 04:30:00
Name: full_date, dtype: datetime64[ns]

In [49]:
row

lagged_1                                 39406.0
lagged_2                                 40331.0
lagged_48                                38131.0
lagged_336                               40865.0
Zone_A                                         1
Zone_B                                         1
Zone_C                                         1
public_holidays                                0
Vacances de la Toussaint                       0
Vacances de Noël                               0
Vacances d'Hiver                               0
Vacances de Printemps                          0
Vacances d'Été                                 1
full_date                    2026-08-23 00:00:00
year                                        2026
month                                          8
hour                                         0.0
day_of_week                                    6
is_weekend                                     1
hour_sin                                     0.0
hour_cos            

### RTE France API

In [86]:
id_client = "863fa354-33b4-4bf6-a844-3dd668062f92"
id_secret = "83b2284d-b40f-4627-916f-d35473030cbf"
url = "https://digital.iservices.rte-france.com/token/oauth/"

response = requests.post(url, auth=(id_client, id_secret))

In [87]:
response.json()

{'access_token': 'k474YPtYGg8Im6fzkvcoIUeWaPI4fzxRjj6ankSZK5EghTijE48cHG',
 'token_type': 'Bearer',
 'expires_in': 3600}

In [88]:
token = response.json()["access_token"]
headers = {
    "Authorization" : f"Bearer {token}"
}

url = "https://digital.iservices.rte-france.com/open_api/consumption/v1/short_term"

data = requests.get(url, headers=headers)

In [89]:
data.json()

{'short_term': [{'type': 'REALISED',
   'start_date': '2026-07-17T00:00:00+02:00',
   'end_date': '2026-07-18T00:00:00+02:00',
   'values': [{'start_date': '2026-07-17T00:00:00+02:00',
     'end_date': '2026-07-17T00:15:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 47387},
    {'start_date': '2026-07-17T00:15:00+02:00',
     'end_date': '2026-07-17T00:30:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 46988},
    {'start_date': '2026-07-17T00:30:00+02:00',
     'end_date': '2026-07-17T00:45:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 45765},
    {'start_date': '2026-07-17T00:45:00+02:00',
     'end_date': '2026-07-17T01:00:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 44762},
    {'start_date': '2026-07-17T01:00:00+02:00',
     'end_date': '2026-07-17T01:15:00+02:00',
     'updated_date': '2026-07-17T13:05:49+02:00',
     'value': 43709},
    {'start_date': '2026-07-17T01

### Open-Meteo API

In [26]:
# ---- Open-Meteo -----
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat_long = {
    '13': {'ville': 'Marseille', 'latitude': 43.2965, 'longitude': 5.3698},     # Bouches-du-Rhône
    '33': {'ville': 'Bordeaux',  'latitude': 44.8378, 'longitude': -0.5792},    # Gironde
    '44': {'ville': 'Nantes',    'latitude': 47.2184, 'longitude': -1.5536},    # Loire-Atlantique
    '59': {'ville': 'Lille',     'latitude': 50.6292, 'longitude': 3.0573},     # Nord
    '69': {'ville': 'Lyon',      'latitude': 45.7640, 'longitude': 4.8357},     # Rhône
    '75': {'ville': 'Paris',     'latitude': 48.8566, 'longitude': 2.3522},     # Paris
}

station_population = {
    '13': 2087658,   # Bouches-du-Rhône 
    '33': 1690493,   # Gironde           
    '44': 1487570,   # Loire-Atlantique   
    '59': 2615635,   # Nord              
    '69': 1914667,   # Rhône             
    '75': 2103778,   # Paris
}

total_pop = sum(station_population.values())
weights = {city: pop / total_pop for city, pop in station_population.items()}


cols = ['T', 'U', 'FF', 'PMER', 'RR1']
df_temp = pd.DataFrame(
    np.zeros(shape=(pred.shape[0], len(cols))),
    columns=cols
)

for k, v in lat_long.items():

    # Calling the API for our department
    params = {
        "latitude": v["latitude"],
        "longitude": v["longitude"],
        "hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure", "wind_speed_10m"],
        "timezone": "Europe/Paris",
        "past_days": 7,
        "forecast_days": 2,
    }
    responses = openmeteo.weather_api(url, params = params)
    
    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_rain = hourly.Variables(2).ValuesAsNumpy()
    hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
    
    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        ).tz_convert(response.Timezone().decode())
    }

    # Preprocessing the weather date
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["rain"] = hourly_rain
    hourly_data["surface_pressure"] = hourly_surface_pressure
    hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
    
    hourly_dataframe = pd.DataFrame(data = hourly_data)

    hourly_dataframe = hourly_dataframe.rename(columns={
        "temperature_2m" : "T",
        "relative_humidity_2m": "U",
        "rain" : "RR1",
        "surface_pressure" : "PMER",
        "wind_speed_10m" : "FF"
    })

    # We add the 30min weather date 
    hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"])
    hourly_dataframe = hourly_dataframe.set_index("date")
    hourly_dataframe = hourly_dataframe.asfreq("30min")

    # We interpolate the missing values
    hourly_dataframe["RR1"] = hourly_dataframe["RR1"].fillna(0)
    hourly_dataframe = hourly_dataframe.interpolate(method="time")
    hourly_dataframe = hourly_dataframe.reset_index()
    hourly_dataframe["date"] = hourly_dataframe["date"].dt.tz_localize(None)
    hourly_dataframe = hourly_dataframe[hourly_dataframe["date"].isin(pred["full_date"])]
    hourly_dataframe = hourly_dataframe.reset_index()
    
    pred[f"{k}T"] = hourly_dataframe[['T']]


    population = weights[k]
    df_temp += hourly_dataframe[cols].values * population


pred = pd.concat([pred, df_temp], axis=1)
dates = pred["full_date"]
pred = pred.drop("full_date", axis=1)

In [27]:
dates

0   2026-08-14 17:30:00
1   2026-08-14 18:00:00
2   2026-08-14 18:30:00
3   2026-08-14 19:00:00
4   2026-08-14 19:30:00
5   2026-08-14 20:00:00
6   2026-08-14 20:30:00
7   2026-08-14 21:00:00
8   2026-08-14 21:30:00
9   2026-08-14 22:00:00
Name: full_date, dtype: datetime64[ns]

In [30]:
pred

,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,33T,44T,59T,69T,75T,T,U,FF,PMER,RR1
0,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,33.520500,37.662003,38.555496,37.010998,38.964996,36.414098,27.603731,9.455471,1008.969048,0.000000
1,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,33.295502,37.687000,38.580498,37.060997,38.814999,36.231932,28.978094,11.134572,1008.732994,0.000000
2,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,32.845501,37.512001,36.805496,36.135998,38.589996,35.466488,31.606397,10.991057,1008.694611,0.000000
3,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,32.395500,37.336998,35.030499,35.210999,38.364998,34.701046,34.234702,10.847543,1008.656250,0.000000
4,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,31.220501,37.024498,33.955498,34.260998,38.214996,34.089264,35.497151,10.827381,1008.704376,0.000000
5,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,30.045500,36.712002,32.880501,33.310997,38.064999,33.477484,36.759603,10.807219,1008.752464,0.000000
6,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,28.995499,35.049500,31.855499,32.335999,37.564999,32.618124,38.113209,10.547158,1008.850937,0.000000
7,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,27.945499,33.387001,30.830500,31.361000,37.064999,31.758765,39.466816,10.287097,1008.949432,0.000000
8,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,26.795500,32.018250,30.330500,30.160999,35.639999,30.747673,43.116406,10.871092,1009.324738,0.000000
9,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,25.645500,30.649500,29.830500,28.961000,34.215000,29.736582,46.765996,11.455087,1009.700111,0.035358


In [31]:
print(pred.columns)
pred

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'year', 'month', 'hour', 'day_of_week', 'is_weekend',
       'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos',
       'month_sin', 'month_cos', 'rolling_mean_24h', 'rolling_std_24h',
       'rolling_mean_7d', 'rolling_std_7d', 'rolling_max_24h',
       'rolling_min_24h', 'consumption_diff_1', 'consumption_diff_48',
       'consumption_pct_change_1', 'consumption_pct_change_48',
       'season_Spring', 'season_Summer', 'season_Winter', '13T', '33T', '44T',
       '59T', '69T', '75T', 'T', 'U', 'FF', 'PMER', 'RR1'],
      dtype='object')


,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,33T,44T,59T,69T,75T,T,U,FF,PMER,RR1
0,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,33.520500,37.662003,38.555496,37.010998,38.964996,36.414098,27.603731,9.455471,1008.969048,0.000000
1,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,33.295502,37.687000,38.580498,37.060997,38.814999,36.231932,28.978094,11.134572,1008.732994,0.000000
2,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,32.845501,37.512001,36.805496,36.135998,38.589996,35.466488,31.606397,10.991057,1008.694611,0.000000
3,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,32.395500,37.336998,35.030499,35.210999,38.364998,34.701046,34.234702,10.847543,1008.656250,0.000000
4,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,31.220501,37.024498,33.955498,34.260998,38.214996,34.089264,35.497151,10.827381,1008.704376,0.000000
5,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,30.045500,36.712002,32.880501,33.310997,38.064999,33.477484,36.759603,10.807219,1008.752464,0.000000
6,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,28.995499,35.049500,31.855499,32.335999,37.564999,32.618124,38.113209,10.547158,1008.850937,0.000000
7,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,27.945499,33.387001,30.830500,31.361000,37.064999,31.758765,39.466816,10.287097,1008.949432,0.000000
8,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,26.795500,32.018250,30.330500,30.160999,35.639999,30.747673,43.116406,10.871092,1009.324738,0.000000
9,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,25.645500,30.649500,29.830500,28.961000,34.215000,29.736582,46.765996,11.455087,1009.700111,0.035358


In [32]:
pred = fe.interactions_linear(pred)

In [33]:
print(len(pred.columns))
pred

82


,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,is_weekend_x_season_Summer,is_holiday_x_season_Summer,hour_x_season_Winter,is_weekend_x_season_Winter,is_holiday_x_season_Winter,temp_x_humidity,temp_x_wind,humidity_x_wind,HDD,CDD
0,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1005.164966,344.312438,261.006274,0.0,18.414098
1,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1049.932317,403.427036,322.658660,0.0,18.231932
2,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1120.967910,389.814187,347.387713,0.0,17.466488
3,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1187.979948,376.421075,371.362393,0.0,16.701046
4,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1210.071761,369.097441,384.341174,0.0,16.089264
5,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1230.619005,361.798507,397.269087,0.0,15.477484
6,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1243.181396,344.028517,401.986042,0.0,14.618124
7,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1253.417351,326.705508,405.998972,0.0,13.758765
8,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1325.729178,334.260771,468.722401,0.0,12.747673
9,50371.0,50478.0,50891.0,45621.0,1,1,1,0,0,0,...,0,0,0.0,0,0,1390.660890,340.635124,535.708542,0.0,11.736582


In [34]:
pred.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'year', 'month', 'hour', 'day_of_week', 'is_weekend',
       'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos',
       'month_sin', 'month_cos', 'rolling_mean_24h', 'rolling_std_24h',
       'rolling_mean_7d', 'rolling_std_7d', 'rolling_max_24h',
       'rolling_min_24h', 'consumption_diff_1', 'consumption_diff_48',
       'consumption_pct_change_1', 'consumption_pct_change_48',
       'season_Spring', 'season_Summer', 'season_Winter', '13T', '33T', '44T',
       '59T', '69T', '75T', 'T', 'U', 'FF', 'PMER', 'RR1', 'temp_sq',
       'humidity_sq', 'hour_x_is_weekend', 'hour_x_is_holiday', 'hour_x_dow',
       'hour_x_month', 'is_weekend_x_month', 'is_holiday_x_month',
       'hour_x_temp', 'hour_x_humidity', 'hour_x_wind', 'is_weekend_x_temp

In [35]:
models = load("../artifacts/model_artifacts/Ridge_2026-07-30_23-24-53/Ridge_models.joblib")

In [36]:
pred2 = pred.copy()
pred2 = fe.drop_useless(pred2)
pred2.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'year', 'is_weekend', 'hour_sin', 'hour_cos',
       'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', '13T', '33T', '44T', '59T', '69T', '75T', 'T', 'U',
       'FF', 'PMER', 'RR1', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
       'is_weekend_x_temp', 'is_holiday_x_temp', 'month_x

In [37]:
predictions = {}
for i in range(0, 10):
    if i == 0:
        p = models[f"Ridge_{i}"].predict(pred)
        predictions[f"horizon_{i}"] = p[0]
    else:
        p = models[f"Ridge_{i}"].predict(pred2.iloc[i-1:i])
        predictions[f"horizon_{i}"] = p[0]

In [38]:
predictions

{'horizon_0': np.float64(50723.43920746894),
 'horizon_1': np.float64(50627.53644457272),
 'horizon_2': np.float64(50365.260824701785),
 'horizon_3': np.float64(50207.71307930005),
 'horizon_4': np.float64(49374.3542893351),
 'horizon_5': np.float64(49363.97577777126),
 'horizon_6': np.float64(49051.593208759135),
 'horizon_7': np.float64(48843.87713989211),
 'horizon_8': np.float64(48069.22612114818),
 'horizon_9': np.float64(47677.989932486016)}

In [197]:
print(type(pred2.iloc[9]))
print(pred2.iloc[9].shape)

print(type(pred2.iloc[[9]]))
print(pred2.iloc[[9]].shape)

<class 'pandas.core.series.Series'>
(79,)
<class 'pandas.core.frame.DataFrame'>
(1, 79)
